# Cross-Strategy Analysis

Algorithm:
- One loop: same candles, same ACTIVE_TRADE
- Collects each BacktestResult into one DataFrame and sorts.

That's the only fair comparison:
- identical data, costs, and sizing for every strategy
- each one keeps its own assigned exit policy, so you're comparing them as designed

Caveats:
- ML strategy is excluded \
  SwingMLStrategy needs its trained model / injected probabilities, so run it from its own notebook and add its row by hand.
- This is a single-window ranking \
  It is great for a quick "who's ahead on this data," but trust the ordering only after you re-run the same loop inside walk-forward \
  (the single backtest can still flatter an overfit strategy).

In [ ]:
import pandas as pd
from engine.backtester import Backtester
from engine.strategy_configurator import params_for
from engine.trade_configurator import ACTIVE_TRADE
from engine.data_configurator import load_data, ACTIVE
from engine.strategies import (
    LevelBreakoutStrategy, InverseLevelBreakoutStrategy,
    FractalBreakoutStrategy, InverseFractalBreakoutStrategy,
    EMACrossoverStrategy, InverseEMACrossoverStrategy,
    SuperTrendStrategy, InverseSuperTrendStrategy, AdaptiveSuperTrendStrategy,
    ExhaustionReversalStrategy, ImpulseFlagStrategy,
    OrderBlockStrategy, InverseOrderBlockStrategy,
    VWAPBandsStrategy, SwingFlipStrategy,
)


In [ ]:
df  = load_data()              # one dataset for all (the ACTIVE spec)


In [ ]:
STRATEGIES = [
    LevelBreakoutStrategy, InverseLevelBreakoutStrategy,        # engine.level_detector horizontal S/R
    FractalBreakoutStrategy, InverseFractalBreakoutStrategy,    # indicators.detect_swing_* fractal pivots
    EMACrossoverStrategy, InverseEMACrossoverStrategy,
    SuperTrendStrategy, InverseSuperTrendStrategy, AdaptiveSuperTrendStrategy,
    ExhaustionReversalStrategy, ImpulseFlagStrategy,
    OrderBlockStrategy, InverseOrderBlockStrategy,
    VWAPBandsStrategy, SwingFlipStrategy,
]   # SwingMLStrategy needs a trained model — run it from its own notebook


In [ ]:
rows = []
for cls in STRATEGIES:
    strat = cls(params_for(cls.name))                                   # each uses its own exit policy
    r = Backtester(strat, symbol=ACTIVE.symbol,
                   trading_config=ACTIVE_TRADE).run(df, interval=ACTIVE.interval)
    rows.append({
        "strategy": strat.name,
        "trades": r.total_trades,
        "win_%": round(r.win_rate * 100, 1),
        "pnl_bps": round(r.total_pnl_bps, 1),
        "profit_factor": round(r.profit_factor, 2),
        "max_dd_bps": round(r.max_drawdown_bps, 1),
        "return_%": round(r.total_return_pct, 2),
        "final_equity": round(r.final_equity, 2),
    })

leaderboard = pd.DataFrame(rows).sort_values("pnl_bps", ascending=False).reset_index(drop=True)
leaderboard